# Iris Flower Classification — Internship Project

**Goal:** Iris dataset par Logistic Regression aur Decision Tree train karke unki performance compare karna.

### Hinglish mein project ka overview
Is project mein hum flower ke 4 measurements (sepal length/width aur petal length/width) se flower ki species predict karenge. Dataset mein 3 classes hain: setosa, versicolor aur virginica.

**Workflow:**
1. Dataset load aur basic EDA
2. Pairplot se features ke relationships dekhna
3. 80/20 train-test split
4. `StandardScaler` se feature scaling (scaler sirf training data par fit hoga, taaki data leakage na ho)
5. Logistic Regression aur Decision Tree train karna
6. Accuracy, precision, recall, F1-score aur confusion matrix compare karna
7. 2 features use karke decision boundary visualize karna

> Note: Decision Tree ko scaling ki zaroorat nahi hoti, lekin yahan dono models ko same preprocessed data par compare karne ke liye scaled features diye gaye hain. Decision-boundary plot sirf visualization ke liye 2 features use karta hai; main models 4 features use karte hain.

## 1. Libraries install/import

Agar local Jupyter environment mein libraries missing hon, toh install command uncomment karke run kar sakte hain.

In [ ]:
# If needed, run this once:
# %pip install pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Dataset load karna

Hum provided Iris CSV URL ko Pandas se load karenge. Agar URL/network access unavailable ho, toh scikit-learn ka built-in Iris dataset fallback ke roop mein use hoga.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"

try:
    df = pd.read_csv(DATA_URL)
    print("Dataset loaded from provided CSV URL.")
except Exception as error:
    print("CSV URL load nahi hua; sklearn built-in Iris dataset use ho raha hai.")
    from sklearn.datasets import load_iris

    iris = load_iris(as_frame=True)
    df = iris.frame.rename(columns={"target": "species"})
    df["species"] = df["species"].map(dict(enumerate(iris.target_names)))

print("Dataset shape:", df.shape)
display(df.head())

## 3. Basic Exploratory Data Analysis (EDA)

`info()`, missing values aur class distribution check karenge. Isse data ki quality aur target classes ka balance samajh aayega.

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values per column:")
display(df.isnull().sum().to_frame("missing_values"))

print("\\nSpecies distribution:")
display(df["species"].value_counts().to_frame("count"))

print("\\nNumerical summary:")
display(df.describe())

## 4. Pairplot

Pairplot se har numerical feature ka doosre features ke saath relationship aur species-wise separation dekh sakte hain.

In [ ]:
sns.pairplot(df, hue="species", diag_kind="hist", corner=True)
plt.suptitle("Iris Dataset — Pairplot", y=1.02)
plt.show()

## 5. Features aur target define karna

`species` hamara target variable hai; baaki 4 columns input features hain. Agar dataset mein missing values hon toh unhe handle karenge.

In [ ]:
X = df.drop(columns=["species"]).copy()
y = df["species"].copy()

# Numeric columns ensure karna
X = X.apply(pd.to_numeric, errors="coerce")

# Missing values ko column median se fill karna
X = X.fillna(X.median(numeric_only=True))

print("Features:", list(X.columns))
print("Target classes:", sorted(y.unique()))

## 6. Train-test split (80/20)

80% data model training ke liye aur 20% unseen testing ke liye. `stratify=y` ensure karta hai ki train aur test dono mein classes ka proportion lagbhag same rahe.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 7. Feature scaling

`StandardScaler` features ko mean 0 aur standard deviation 1 ke aas-paas scale karta hai. **Important:** scaler ko sirf `X_train` par fit karte hain; `X_test` par sirf transform. Isse test data ki information training process mein leak nahi hoti.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 8. Models train karna

Dono classifiers ko train data par fit karenge. Logistic Regression ek linear classification model hai, jabki Decision Tree feature-based rules/splits banata hai.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=4)
}

predictions = {}

for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions[model_name] = model.predict(X_test_scaled)
    print(f"{model_name} trained successfully.")

## 9. Performance benchmark

Multiclass classification ke liye precision, recall aur F1-score ko `weighted` average ke saath calculate kar rahe hain. Weighted average har class ke sample count ko consider karta hai.

In [ ]:
results = []

for model_name, y_pred in predictions.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (weighted)": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-score (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))

### Classification reports

Neeche har species ke liye precision, recall, F1-score aur support (test samples ki count) diya hai.

In [ ]:
for model_name, y_pred in predictions.items():
    print(f"\\n===== {model_name} =====")
    print(classification_report(y_test, y_pred, zero_division=0))

## 10. Confusion matrix heatmaps

Confusion matrix mein rows actual classes aur columns predicted classes hoti hain. Diagonal values correct predictions hoti hain; off-diagonal values misclassifications.

In [ ]:
class_labels = sorted(y.unique())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (model_name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=class_labels)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_labels, yticklabels=class_labels, ax=ax
    )
    ax.set_title(f"{model_name} — Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")

plt.tight_layout()
plt.show()

## 11. Decision boundary visualization (2 features)

2D plot banane ke liye hum `petal_length` aur `petal_width` use karenge. Ek alag Logistic Regression aur Decision Tree pipeline train hogi, kyunki 2D graph mein sirf 2 features hi plot ho sakte hain.

**Important:** Yeh visualization-only models hain. Upar benchmark kiye gaye main models 4 features use karte hain.

In [ ]:
# Seaborn Iris CSV mein feature names underscore format mein hote hain.
# Built-in fallback mein bhi same feature names hain.
feature_candidates = ["petal_length", "petal_width"]
if not all(col in X.columns for col in feature_candidates):
    print("Expected petal feature columns nahi mile. Available columns:", list(X.columns))
else:
    X_2d = X[feature_candidates]

    X2_train, X2_test, y2_train, y2_test = train_test_split(
        X_2d, y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y
    )

    boundary_models = {
        "Logistic Regression (2 features)": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        ),
        "Decision Tree (2 features)": DecisionTreeClassifier(
            random_state=RANDOM_STATE, max_depth=4
        )
    }

    x_min, x_max = X_2d.iloc[:, 0].min() - 0.5, X_2d.iloc[:, 0].max() + 0.5
    y_min, y_max = X_2d.iloc[:, 1].min() - 0.5, X_2d.iloc[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )
    grid = pd.DataFrame(
        np.c_[xx.ravel(), yy.ravel()],
        columns=feature_candidates
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, (model_name, model) in zip(axes, boundary_models.items()):
        model.fit(X2_train, y2_train)
        grid_predictions = model.predict(grid)

        # Classes ko numeric codes mein map karke contour plot
        class_to_num = {label: i for i, label in enumerate(class_labels)}
        Z = pd.Series(grid_predictions).map(class_to_num).to_numpy().reshape(xx.shape)

        ax.contourf(xx, yy, Z, alpha=0.25, levels=np.arange(len(class_labels) + 1) - 0.5)
        sns.scatterplot(
            data=pd.DataFrame({
                feature_candidates[0]: X2_test.iloc[:, 0].values,
                feature_candidates[1]: X2_test.iloc[:, 1].values,
                "species": y2_test.values
            }),
            x=feature_candidates[0], y=feature_candidates[1],
            hue="species", palette="deep", edgecolor="black",
            s=55, ax=ax
        )
        ax.set_title(model_name)
        ax.set_xlabel("Petal length (cm)")
        ax.set_ylabel("Petal width (cm)")

    plt.tight_layout()
    plt.show()

## 12. Result interpretation

Neeche wali table ke metrics ko compare karke dekhein. Higher score generally better hota hai, lekin model selection mein problem context aur class-wise report bhi important hain.

- **Accuracy:** Total predictions mein se kitni correct hain.
- **Precision:** Model ne kisi class ko predict kiya, unmein se kitni predictions correct thi.
- **Recall:** Actual class ke samples mein se model ne kitne identify kiye.
- **F1-score:** Precision aur recall ka harmonic mean.

Final result ko apne notebook ke output ke basis par likhein; scores ko manually assume ya hard-code na karein.

In [ ]:
best_model_by_f1 = results_df["F1-score (weighted)"].idxmax()
print("Best model by weighted F1-score:", best_model_by_f1)
display(results_df.sort_values("F1-score (weighted)", ascending=False).round(4))

## 13. Conclusion (fill after running)

Notebook run karne ke baad apne actual output ke basis par yeh points complete karein:

1. Is experiment mein best weighted F1-score kis model ka aaya?
2. Dono models ki accuracy mein kya difference tha?
3. Confusion matrix mein kaunsi species sabse achhi classify hui?
4. Kya 2D decision boundary plot se classes ka separation samajh aaya?

**Reproducibility:** `random_state=42` use kiya gaya hai, jisse same environment/data ke saath split repeatable rahega.